# Исследование развития игровой индустрии 

### Цели и задачи проекта

Цель проекта - привлечь новую аудиторию к игре с помощью публикации статьи о развитии игровой индустрии в начале XXI века.

Задачи проекта:
1. Исследовать ключевые тенденции развития игровой индустрии 2000 - 2013 гг.
2. Определить наиболее популярные игровые платформы.
3. Выявить наиболее востребованные игровые жанры.
4. Исследовать региональные предпочтения игроков.
5. Подготовить аналитические выводы для статьи.

### Описание данных
Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, 
а также пользовательские и экспертные оценки игр:

* Name — название игры.
* Platform — название платформы.
* Year of Release — год выпуска игры.
* Genre — жанр игры.
* NA sales — продажи в Северной Америке (в миллионах проданных копий).
* EU sales — продажи в Европе (в миллионах проданных копий).
* JP sales — продажи в Японии (в миллионах проданных копий).
* Other sales — продажи в других странах (в миллионах проданных копий).
* Critic Score — оценка критиков (от 0 до 100).
* User Score — оценка пользователей (от 0 до 10).
* Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

1. Загрузка данных и знакомство с ними.
2. Проверка ошибок в данных и их предобработка.
3. Фильтрация данных.
4. Категоризация данных.
5. Итоговый вывод
---

## 1. Загрузка данных и знакомство с ними

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

In [3]:
# Выводим данные о датасете
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Количество строк до очистки данных
quantity_str = len(df)

**Вывод по результатам первичного знакомства с данными**

По результатам первичного знакомства с данными можно сделать следующие выводы:

1. Датасет содержит **16 956 записей** и **10 колонок**, что является достаточным объёмом данных для проведения исследования.

2. Названия и количество столбцов соответствуют описанию датасета.

3. Содержимое столбцов соответствует их назначению и не вызывает явных вопросов на этапе первичного ознакомления.

4. В датасете присутствуют данные двух типов:

   * `float64` — 4 столбца;
   * `object` — 7 столбцов.

5. В ряде столбцов наблюдаются некорректные типы данных:

   * `EU sales`, `JP sales` и `User Score` содержат числовые показатели, однако имеют тип `object`;
   * после дополнительной проверки данные столбцы необходимо привести к числовому формату.

6. В столбце `Year of Release` содержатся значения года выпуска игр. Тип данных `float64`, вероятно, связан с наличием пропущенных значений. После обработки пропусков столбец можно привести к числовому для удбной рабы с этим столбцом.

7. В датасете присутствуют пропущенные значения:

   * `Name` — 2 пропуска;
   * `Year of Release` — 275 пропусков;
   * `Genre` — 2 пропуска;
   * `Critic Score` — 8 714 пропусков (51%);
   * `User Score` — 6 804 пропуска (40%);
   * `Rating` — 6 871 пропуск (41%).

**Предположения о причинах пропусков**

**Name (2 пропуска)**

Возможной причиной могут быть ошибки при сборе данных, повреждение записей при выгрузке или отсутствие информации об отдельных играх в источнике данных. Поскольку количество таких записей минимально, их влияние на анализ будет незначительным.

**Genre (2 пропуска)**

Вероятно, данные о жанре отсутствуют по тем же причинам, что и название игры. Возможно, речь идёт о неполных или некорректно загруженных записях.

**Critic Score (51% пропусков)**

Более половины игр не имеют оценки критиков. Возможные причины:

* часть игр не получила профессиональных обзоров;
* данные собирались из разных источников, где оценки критиков были доступны не для всех игр;
* многие старые или малоизвестные игры не были оценены специализированными изданиями.

**User Score (40% пропусков)**

Отсутствие пользовательских оценок может быть связано с:

* низкой популярностью отдельных игр;
* недостаточным количеством пользовательских отзывов;
* отсутствием информации в используемых источниках данных.

**Rating (41% пропусков)**

Пропуски могут возникать по следующим причинам:

* для части игр возрастной рейтинг не присваивался;
* некоторые игры выпускались за пределами регионов действия;
* информация о рейтинге могла отсутствовать в исходных данных.




---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма


In [5]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [6]:
# Приводим столбцы к стилю snake case 
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [7]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных

**Предположения причин**

Столбцы **eu_sales**, **jp_sales** и **user_score** имеют тип **object**, что может быть связано с наличием текстовых значений или хранением числовых данных в строковом формате.

Столбец **year_of_release** имеет тип **float64** из-за наличия пропущенных значений, столбец с целыми числами при наличии **NaN** автоматически преобразуется в float64.

Столбец **year_of_release** переводим к типу **Int64**, поскольку содержит информацию о годе выпуска игр. Использование целочисленного типа данных является более корректным, а тип **Int64** позволяет сохранять пропущенные значения.

In [8]:
# Переводим в целочисленный тип данных Int64
df['year_of_release'] = df['year_of_release'].astype('Int64', errors='ignore')

In [9]:
# Выводим результат
df['year_of_release'].head(5)

0    2006
1    1985
2    2008
3    2009
4    1996
Name: year_of_release, dtype: Int64

In [10]:
# Создаем цикл для перевода в числовой тип с плавающей точкой float64
for column in ['eu_sales', 'jp_sales', 'user_score']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

In [11]:
# Выводим результат
print(df[['eu_sales', 'jp_sales', 'user_score']].head(5))
print('-'*35)
print(df[['eu_sales', 'jp_sales', 'user_score']].dtypes)

   eu_sales  jp_sales  user_score
0     28.96      3.77         8.0
1      3.58      6.81         NaN
2     12.76      3.79         8.3
3     10.93      3.28         8.0
4      8.89     10.22         NaN
-----------------------------------
eu_sales      float64
jp_sales      float64
user_score    float64
dtype: object


### 2.3. Наличие пропусков в данных

In [12]:
# Выводим сумму пропусков и их долю
missing_table = pd.DataFrame({
    'Количество пропусков': df.isnull().sum(),
    'Доля пропусков (%)': (df.isnull().mean() * 100).round(2)
})

missing_table = missing_table.sort_values('Количество пропусков', ascending=False)
missing_table

,Количество пропусков,Доля пропусков (%)
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


**Основные наблюдения**

Анализ пропусков показал, что наибольшее количество отсутствующих значений содержится в столбцах:

*Основные пропуски*

- user_score — 9 268 пропусков (55%)
- critic_score — 8 714 пропусков (51%) 
- rating — 6 871 пропуск (41%).

*Не основные пропуски*

- year_of_release 275 пропусков (2%) 
- name и genre  по 2 пропуска 
- eu_sales и jp_sales — 6 и 4 пропуска менее 1% данных. 

**Предположения о причинах пропусков**

- В user_score и critic_score часть игр могла не получить пользовательских или экспертных оценок.
- В rating пропуски, скорее всего, связаны с тем, что часть игр не получила возрастной рейтинг ESRB или выпускалась в регионах, где он не используется.
- В year_of_release, name и genre пропуски, вероятно, возникли из-за неполноты или ошибок при сборе данных.
- В eu_sales и jp_sales пропуски могут быть связаны с отсутствием данных по отдельным регионам или ошибками выгрузки.

**Обработка пропусков**

- Пропуски в user_score и critic_score оставлены без изменений, поскольку заполнение отсутствующих значений может исказить реальные оценки пользователей и критиков.
- Строки с пропусками в name и genre будут удалены, так как они составляют незначительную долю и содержат ключевые признаки для анализа.
- Строки с пропусками в year_of_release также будут удалены, так как данный признак важен для анализа 2000–2013 годов, а пустые значения буду искажать данные так как не определены за какой период они отображают информацию.
- Пропуски в eu_sales и jp_sales будут заполнены с использованием среднего значения, рассчитанного по группам platform и year_of_release.
- Пропуски в столбце rating будут заменены на значение "Нет рейтинга", чтобы явно обозначить отсутствие возрастной классификации.


In [13]:
# Удаление строк со столбсов 'name','genre', 'year_of_release'.
df = df.dropna(subset = ['name','genre', 'year_of_release'])

In [14]:
# Функция для заралнения пропусков
def mean_sales(row, column):
    if pd.isna(row[column]):
        group = df[(df['platform'] == row['platform']) &
                (df['year_of_release']==row['year_of_release'])]
        return group[column].mean()
    else:
        return row[column]

In [15]:
# Записываем в колонку 'eu_sales'
df['eu_sales'] = df.apply(mean_sales, axis= 1, column='eu_sales')

In [16]:
# Записываем в колонку 'jp_sales'
df['jp_sales'] = df.apply(mean_sales, axis= 1, column='jp_sales')

In [17]:
# Заменяем пропуски столбца 'rating' на 'No rating'
df['rating'] = df['rating'].fillna('No rating')

In [18]:
# Проверяем пропуски 
df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating                0
dtype: int64

Остались пропуски только а колонках **critic_score** и **user_score** как планировали.

### 2.4. Явные и неявные дубликаты в данных

In [19]:
# Выводим уникальные значения столбцов 'platform', 'year_of_release', 'genre', 'rating' через цикл
for column in ['platform', 'year_of_release','genre', 'rating']:
    print(f'Столбец {column}:')
    print()
    print(df[column].unique())
    print()

Столбец platform:

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']

Столбец year_of_release:

<IntegerArray>
[2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004,
 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994,
 1982, 2016, 2003, 1986, 2000, 1995, 1991, 1981, 1987, 1980, 1983]
Length: 37, dtype: Int64

Столбец genre:

['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']

Столбец rating:

['E' 'No rating' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']



In [20]:
# Переводим в нижний регистр столбцы 'platform' и 'genre'
for column in ['platform', 'genre']:
    df[column] = df[column].str.lower()

In [21]:
# Переводим к верхнему регистру столбец 'rating'
df['rating'] = df['rating'].str.upper()

In [22]:
# Проверяем результат
for column in ['platform', 'year_of_release','genre', 'rating']:
    print(f'Столбец {column}:')
    print()
    print(df[column].unique())
    print(df[column].dtype)
    print()

Столбец platform:

['wii' 'nes' 'gb' 'ds' 'x360' 'ps3' 'ps2' 'snes' 'gba' 'ps4' '3ds' 'n64'
 'ps' 'xb' 'pc' '2600' 'psp' 'xone' 'wiiu' 'gc' 'gen' 'dc' 'psv' 'sat'
 'scd' 'ws' 'ng' 'tg16' '3do' 'gg' 'pcfx']
object

Столбец year_of_release:

<IntegerArray>
[2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004,
 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994,
 1982, 2016, 2003, 1986, 2000, 1995, 1991, 1981, 1987, 1980, 1983]
Length: 37, dtype: Int64
Int64

Столбец genre:

['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']
object

Столбец rating:

['E' 'NO RATING' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
object



**Вывод о наличии неявных дубликатов**

Анализ уникальных значений столбцов platform, year_of_release, genre и rating показал отсутствие неявных дубликатов, связанных с опечатками, разным регистром или различными вариантами написания. Все категориальные значения представлены в едином формате и не требуют дополнительной нормализации. Значение NO RATING в столбце rating является специально добавленным индикатором отсутствия возрастного рейтинга и не относится к неявным дубликатам.

In [23]:
# Узнаем количество дублирующих строк 
df.duplicated().sum()

np.int64(235)

In [24]:
# Выводм на экран дублирующие строки
df[df.duplicated()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
268,Batman: Arkham Asylum,ps3,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
368,James Bond 007: Agent Under Fire,ps2,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
717,God of War: Ascension,ps3,2013,action,1.23,0.63,0.04,0.35,80.0,7.5,M
823,Wipeout: The Game,wii,2009,misc,1.94,0.00,0.00,0.12,NaN,NaN,NO RATING
848,Rayman Raving Rabbids: TV Party,wii,2008,misc,0.72,1.08,0.00,0.23,73.0,7.7,E10+
...,...,...,...,...,...,...,...,...,...,...,...
16671,Fullmetal Alchemist: Prince of the Dawn,wii,2009,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NO RATING
16753,Routes PE,ps2,2007,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NO RATING
16799,Transformers: Prime,wii,2012,action,0.00,0.01,0.00,0.00,NaN,NaN,NO RATING
16912,Metal Gear Solid V: The Definitive Experience,xone,2016,action,0.01,0.00,0.00,0.00,NaN,NaN,M


**Вывод о явных дубликатах** 

Проверка показала наличие 235 явных дубликатов. Поскольку данные строки полностью повторяют существующие записи, они будут удалены из набора данных для корректного дальнейшего анализа.

In [25]:
#Удоляем явные дубликаты и уставляем в текущем датасете
df.drop_duplicates(inplace=True)

In [26]:
# Преверяем удоление
df.duplicated().sum()

np.int64(0)

In [27]:
# Количество строк после очистки данных
quantity_str_cleaned = len(df)

Все явные дубликаты удалились как и планировалось.

In [28]:
# Число строк до очистки данных
quantity_str

16956

In [29]:
# Число строк после очистки данных
quantity_str_cleaned

16444

In [30]:
# Разница строк в абсолютном значении
quantity_str - quantity_str_cleaned

512

In [31]:
# Разница строк в относительном значении
f'{round((1- quantity_str_cleaned / quantity_str) * 100, 2)} %'

'3.02 %'

**Вывод после обработки данных в датасете**

В ходе предобработки были приведены к корректным типы данных, обработаны пропуски, проверены неявные дубликаты и удалены 235 явных дубликатов. Также были удалены строки с критически важными пропусками. В результате количество строк сократилось с 16 956 до 16 444, было удалено 512 строк (3% от исходного объёма данных). Данные подготовлены для дальнейшего исследовательского анализа.

---

## 3. Фильтрация данных

In [32]:
# Делаем фильтрацию по годам с 2000 по 2013 и фиксируем в переменную df_actual
df_actual = df[df['year_of_release'].between(2000, 2013)].sort_values(by='year_of_release')

In [33]:
# Выводим на экран результат 
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
2743,Mega Man X5,ps,2000,platform,0.30,0.21,0.20,0.05,76.0,8.9,E
6558,Star Trek: Invasion,ps,2000,simulation,0.15,0.10,0.00,0.02,76.0,NaN,E
11063,ESPN NBA 2Night,ps2,2000,sports,0.05,0.04,0.00,0.01,62.0,NaN,E
6586,Star Wars Episode I: Battle for Naboo,n64,2000,simulation,0.21,0.05,0.00,0.00,NaN,NaN,NO RATING
1726,Mario Tennis,gb,2000,sports,0.50,0.18,0.44,0.06,NaN,NaN,NO RATING
...,...,...,...,...,...,...,...,...,...,...,...
724,Assassin's Creed IV: Black Flag,xone,2013,action,1.48,0.55,0.00,0.21,NaN,7.4,M
16175,Storm Lover 2nd,psp,2013,misc,0.00,0.00,0.02,0.00,NaN,NaN,NO RATING
727,Madden NFL 25,x360,2013,sports,1.98,0.06,0.00,0.19,80.0,5.6,E
16162,White Album 2: Shiawase no Mukougawa,psv,2013,adventure,0.00,0.00,0.02,0.00,NaN,NaN,NO RATING


In [34]:
# Проверяем уникальные значения
df_actual['year_of_release'].unique()

<IntegerArray>
[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
 2013]
Length: 14, dtype: Int64

---

## 4. Категоризация данных

In [35]:
# Делаем категоризацию по оценкам пользователей через pd.cut
df_actual['user_score_category'] = pd.cut(df_actual['user_score'], 
                                     bins=[0, 3, 8, 10.1], 
                                     labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], 
                                     right=False)

In [36]:
# Меняем значение NaN на 'нет оценки'
df_actual['user_score_category'] = df_actual['user_score_category'].astype(str).fillna('нет оценки')

In [37]:
# Выводим уникальные значения
df_actual['user_score_category'].unique()

array(['высокая оценка', np.str_('nan'), 'средняя оценка',
       'низкая оценка'], dtype=object)

In [38]:
# Выводим столбец'score_category'
df_actual['user_score_category']

2743     высокая оценка
6558                nan
11063               nan
6586                nan
1726                nan
              ...      
724      средняя оценка
16175               nan
727      средняя оценка
16162               nan
6777     средняя оценка
Name: user_score_category, Length: 12781, dtype: object

In [39]:
# Делаем категоризацию по оценкам критиков через pd.cut
df_actual['critic_score_category'] = pd.cut(df_actual['critic_score'], 
                                     bins=[0, 30, 80, 100.1], 
                                     labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], 
                                     right=False)

In [40]:
# Меняем значение NaN на 'нет оценки'
df_actual['critic_score_category'] = df_actual['critic_score_category'].astype(str).fillna('нет оценки')

In [41]:
# Выводим уникальные значения
df_actual['critic_score_category'].unique()

array(['средняя оценка', np.str_('nan'), 'высокая оценка',
       'низкая оценка'], dtype=object)

In [42]:
# Выводим столбец'critic_score'
df_actual['critic_score_category']

2743     средняя оценка
6558     средняя оценка
11063    средняя оценка
6586                nan
1726                nan
              ...      
724                 nan
16175               nan
727      высокая оценка
16162               nan
6777     средняя оценка
Name: critic_score_category, Length: 12781, dtype: object

In [43]:
# Находим количество игр для выделеных категорий 'score_category','critic_score'
for column in ['user_score_category', 'critic_score_category']:
    print(f'Количество игр по столбцу: {column}')
    print(df_actual.groupby(column)['name'].count())
    print()

Количество игр по столбцу: user_score_category
user_score_category
nan               6298
высокая оценка    2286
низкая оценка      116
средняя оценка    4081
Name: name, dtype: int64

Количество игр по столбцу: critic_score_category
critic_score_category
nan               5612
высокая оценка    1692
низкая оценка       55
средняя оценка    5422
Name: name, dtype: int64



In [44]:
# Групируем и выводим топ 7 платформ по количеству игр
gruop_platform = df_actual.groupby('platform')['name'].count().reset_index()

In [45]:
df_actual['platform'].value_counts().head(7)

platform
ps2     2127
ds      2120
wii     1275
psp     1180
x360    1121
ps3     1087
gba      811
Name: count, dtype: int64

In [46]:
# Итоговый датасет
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_category,critic_score_category
2743,Mega Man X5,ps,2000,platform,0.30,0.21,0.20,0.05,76.0,8.9,E,высокая оценка,средняя оценка
6558,Star Trek: Invasion,ps,2000,simulation,0.15,0.10,0.00,0.02,76.0,NaN,E,nan,средняя оценка
11063,ESPN NBA 2Night,ps2,2000,sports,0.05,0.04,0.00,0.01,62.0,NaN,E,nan,средняя оценка
6586,Star Wars Episode I: Battle for Naboo,n64,2000,simulation,0.21,0.05,0.00,0.00,NaN,NaN,NO RATING,nan,nan
1726,Mario Tennis,gb,2000,sports,0.50,0.18,0.44,0.06,NaN,NaN,NO RATING,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,Assassin's Creed IV: Black Flag,xone,2013,action,1.48,0.55,0.00,0.21,NaN,7.4,M,средняя оценка,nan
16175,Storm Lover 2nd,psp,2013,misc,0.00,0.00,0.02,0.00,NaN,NaN,NO RATING,nan,nan
727,Madden NFL 25,x360,2013,sports,1.98,0.06,0.00,0.19,80.0,5.6,E,средняя оценка,высокая оценка
16162,White Album 2: Shiawase no Mukougawa,psv,2013,adventure,0.00,0.00,0.02,0.00,NaN,NaN,NO RATING,nan,nan


---

## 5. Итоговый вывод

В ходе работы был выполнен анализ данных о продажах видеоигр. На этапе предобработки были приведены к корректным типы данных, обработаны пропущенные значения, проверены неявные дубликаты и удалены 235 явных дубликатов. В результате количество строк сократилось с 16 956 до 16 444, что составляет около 3% от исходного объёма данных.

Для дальнейшего анализа был сформирован отдельный срез данных **df_actual**, содержащий игры, выпущенные в период с 2000 по 2013 год включительно. Данный период был выбран в соответствии с требованиями исследования.

В исходный датасет были добавлены новые поля:

* **user_score_category** — категория пользовательской оценки игры;
* **critic_score_category** — категория оценки игры критиками.

Категоризация была выполнена по четырём группам: низкая оценка, средняя оценка, высокая оценка и «нет оценки».

Распределение игр по категориям пользовательских оценок показало, что большинство игр не имеют пользовательской оценки 6 298 игр. Среди игр с оценками преобладает категория со средней оценкой 4 081 игра, далее следуют игры с высокой оценкой 2 286 игр. Низкую пользовательскую оценку получили 116 игр.

Распределение игр по категориям оценок критиков показало схожую картину. Наибольшее количество игр относится к категории со средней оценкой 5 422 игры, при этом для 5 612 игр оценки критиков отсутствуют. Высокую оценку критиков получили 1 692 игры, а низкую — 55 игр.

Также был проведён анализ количества игр по платформам за рассматриваемый период. В топ-7 платформ по количеству выпущенных игр вошли: PS2 (2 127 игр), DS (2 120 игр), Wii (1 275 игр), PSP (1 180 игр), X360 (1 121 игра), PS3 (1 087 игр) и GBA (811 игр).

Таким образом, данные были очищены от пропусков и дубликатов, дополнены новыми аналитическими признаками и подготовлены для дальнейшего исследовательского анализа. Основная часть игр относится к категории средних оценок пользователей и критиков, при этом значительная доля игр не имеет оценок.

